# Part 1: The Coding Implementation 

Library Usage

In [ ]:
import nltk
import random

## Step 1: Data Acquisition and Splitting

Use the Natural Language Toolkit (nltk) to download the Brown Corpus using the universal tagset.

In [7]:
import nltk
import random

# Download required NLTK data
nltk.download("brown")
nltk.download("universal_tagset")

from nltk.corpus import brown

[nltk_data] Downloading package brown to /home/super/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/super/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


### A. import and split data 

In [8]:
# TODO : Load the corpus and split it into 80% training data and 20% testing data.

# Load Brown Corpus sentences with universal POS tags
sentences = brown.tagged_sents(tagset="universal")

# Convert to a normal list so we can shuffle and split it
sentences = list(sentences)

print("Number of sentences:", len(sentences))
print("First sentence:")
print(sentences[0])

Number of sentences: 57340
First sentence:
[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


In [9]:
# Shuffle the sentences so training and testing data are mixed fairly
random.seed(42)
random.shuffle(sentences)

# 80% for training, 20% for testing
split_index = int(0.8 * len(sentences))

train_data = sentences[:split_index]
test_data = sentences[split_index:]

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 45872
Testing sentences: 11468


## Step 2: Parameter Estimation & Laplace Smoothing

Train your HMM by estimating the initial ($\pi$), transition ($A$), and emission ($B$) probabilities using Maximum Likelihood Estimation (MLE) based on frequency counts in your training set.

To prevent zero-probabilities when encountering unseen words during testing, apply **Laplace (Add-$\alpha$) Smoothing** to your emission matrix. The smoothed probability of emitting word $o_t$ given tag $s_i$ is:

$$P_{\text{smoothed}}(o_t|s_i) = \frac{\text{Count}(o_t \text{ tagged as } s_i) + \alpha}{\text{Count}(s_i) + \alpha|V|}$$

Where $\alpha = 1.0$ and $|V|$ is the total number of unique words in your training vocabulary. Ensure you create a specific dictionary key (e.g., `"<OOV>"`) to store the probability mass for unseen words.



In [4]:
from collections import defaultdict, Counter

# Count containers
initial_counts = Counter()                  # Count first tag in each sentence
transition_counts = defaultdict(Counter)    # Count tag_i -> tag_j
emission_counts = defaultdict(Counter)      # Count tag -> word
tag_counts = Counter()                      # Count each tag
vocabulary = set()                          # Unique words

# Go through each sentence in training data
for sentence in train_data:
    if len(sentence) == 0:
        continue

    # First word and first tag
    first_word, first_tag = sentence[0]
    initial_counts[first_tag] += 1

    # Count every word/tag pair
    for word, tag in sentence:
        word = word.lower()      # Make words lowercase
        emission_counts[tag][word] += 1
        tag_counts[tag] += 1
        vocabulary.add(word)

    # Count tag transitions
    for i in range(len(sentence) - 1):
        current_tag = sentence[i][1]
        next_tag = sentence[i + 1][1]
        transition_counts[current_tag][next_tag] += 1

# Get all POS tags
tags = sorted(tag_counts.keys())

print("Number of tags:", len(tags))
print("Tags:", tags)
print("Vocabulary size:", len(vocabulary))

print("\nInitial tag counts:")
print(initial_counts)

print("\nExample emission counts for NOUN:")
print(emission_counts["NOUN"].most_common(10))

Number of tags: 12
Tags: ['.', 'ADJ', 'ADP', 'ADV', 'CONJ', 'DET', 'NOUN', 'NUM', 'PRON', 'PRT', 'VERB', 'X']
Vocabulary size: 45755

Initial tag counts:
Counter({'DET': 10665, 'NOUN': 6800, 'PRON': 6385, 'ADP': 6261, 'ADV': 4216, '.': 2931, 'CONJ': 2231, 'VERB': 2139, 'ADJ': 1800, 'PRT': 1555, 'NUM': 869, 'X': 20})

Example emission counts for NOUN:
[('time', 1286), ('af', 994), ('man', 879), ('years', 843), ('state', 772), ('mr.', 752), ('world', 727), ('people', 726), ('way', 698), ('men', 635)]


## Step 3: The Log-Space Viterbi Decoder

Because standard probabilities are strictly between 0 and 1, multiplying them together for a normal-length sentence results in infinitesimally small numbers (Underflow). Transform the recursive Viterbi step into log-space:

$$\log v_t(j) = \max_{i=1}^N \left[ \log v_{t-1}(i) + \log P(s_j|s_i) \right] + \log P(o_t|s_j)$$

*Hint: $\log(0)$ is mathematically undefined. Add a tiny constant (epsilon, $10^{-12}$) to all probabilities before taking the logarithm to avoid math domain errors.*


In [5]:
import math

def log_viterbi(sentence, tags, A, B, pi):
    epsilon = 1e-12  # Small value to avoid log(0)

    # TODO: Initialize dynamic programming tables
    viterbi = []      # Stores the best log-probability for each tag at each word position
    backpointer = []  # Stores the best previous tag, so we can recover the final tag sequence

    first_word = sentence[0].lower()  # Convert word to lowercase to match training vocabulary

    viterbi.append({})      # Table for position 0
    backpointer.append({})  # Backpointer for position 0

    # Initialize the first word
    for tag in tags:
        # Get emission probability P(first_word | tag)
        # If the word is unseen, use <OOV> probability
        emission_prob = B[tag].get(first_word, B[tag]["<OOV>"])

        # log probability = log(initial tag probability) + log(emission probability)
        viterbi[0][tag] = (
            math.log(pi.get(tag, epsilon) + epsilon)
            + math.log(emission_prob + epsilon)
        )

        # First word has no previous tag
        backpointer[0][tag] = None

    # TODO: Perform the forward pass in log-space
    for t in range(1, len(sentence)):
        word = sentence[t].lower()  # Current word

        viterbi.append({})      # Table for current position
        backpointer.append({})  # Backpointer for current position

        for current_tag in tags:
            # Get emission probability P(word | current_tag)
            emission_prob = B[current_tag].get(word, B[current_tag]["<OOV>"])

            best_score = float("-inf")  # Best log-probability found so far
            best_previous_tag = None    # Best previous tag found so far

            for previous_tag in tags:
                # Score =
                # previous best score
                # + log transition probability P(current_tag | previous_tag)
                # + log emission probability P(word | current_tag)
                score = (
                    viterbi[t - 1][previous_tag]
                    + math.log(A[previous_tag].get(current_tag, epsilon) + epsilon)
                    + math.log(emission_prob + epsilon)
                )

                # Keep the best previous tag
                if score > best_score:
                    best_score = score
                    best_previous_tag = previous_tag

            # Save best score and best previous tag
            viterbi[t][current_tag] = best_score
            backpointer[t][current_tag] = best_previous_tag

    # TODO: Backtrack to find the optimal sequence of tags

    # Find the best tag at the last word
    best_last_tag = max(viterbi[-1], key=viterbi[-1].get)

    # Start the final tag sequence from the last tag
    best_tags = [best_last_tag]

    # Move backward from the last word to the first word
    for t in range(len(sentence) - 1, 0, -1):
        best_last_tag = backpointer[t][best_last_tag]
        best_tags.insert(0, best_last_tag)

    return best_tags

## Step 4: Evaluation

Write an evaluation loop to process the 20% test sentences and compute your total word-level accuracy.

$$\text{Accuracy} = \frac{\text{Total Correctly Tagged Words}}{\text{Total Words in Test Set}}$$


In [6]:
# Step 4: Evaluation
# Goal: compute word-level accuracy on the 20% test set

total_correct = 0  # Count how many words get the correct POS tag
total_words = 0    # Count total words in the test set

# Go through each sentence in the test data
for sentence in test_data:
    if len(sentence) == 0:
        continue

    # Separate words and true tags
    words = [word for word, tag in sentence]
    true_tags = [tag for word, tag in sentence]

    # Predict tags using our HMM model
    predicted_tags = log_viterbi(words, tags, A, B, pi)

    # Compare predicted tags with true tags
    for true_tag, predicted_tag in zip(true_tags, predicted_tags):
        if true_tag == predicted_tag:
            total_correct += 1

        total_words += 1

# Calculate accuracy
accuracy = total_correct / total_words

print("Total correctly tagged words:", total_correct)
print("Total words in test set:", total_words)
print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

NameError: name 'A' is not defined